## Moving from Output Parsers to Runnables in LangChain

We recently covered **Output Parsers**, components that help us structure and clean the raw text coming from LLMs.  
Now we move towards **Runnables**, which are the backbone of LangChain’s execution model.

---

### The Problem Before Runnables
Originally, LangChain had multiple components:
- Prompt templates (`.format()`)
- LLMs (`.predict()`)
- Retrievers (`.get_relevant_documents()`)
- Parsers (`.parse()`)

Each had its **own API** with different method names.  
This inconsistency made it difficult to connect them into flexible workflows. Developers had to write custom glue code to pass outputs from one component to another, which slowed down development and reduced reusability.

---

### How Runnables Solved the Problem
LangChain introduced **Runnables** as a **standardized interface**.  
Every component now behaves the same way, exposing methods like:
- `.invoke()` → run once with input  
- `.batch()` → run multiple inputs at once  
- `.stream()` → stream partial outputs  

This means:
- Any component can be chained together without custom integration.  
- Data flows automatically from one step to the next.  
- Workflows become modular, reusable, and easier to debug.  

---

### How Runnables Work in Web Applications
In a real web app:
- **PromptTemplate Runnable** formats user input.  
- **LLM Runnable** generates a response.  
- **Parser Runnable** structures the output.  
- **Retriever Runnable** fetches context documents.  
- **Runnable Primitives** orchestrate the flow (sequential, parallel, conditional).  

This allows developers to build pipelines that are:
- **Fast** (parallel execution reduces latency)  
- **Interactive** (streaming responses improve UX)  
- **Flexible** (custom logic via RunnableLambda)  
- **Dynamic** (branching workflows adapt at runtime)  

---

### Categories of Runnables (2026)

#### 1. Task-Specific Runnables (the workers)
- **PromptTemplate** – formats input into a prompt  
- **LLM / ChatModel** – generates a response  
- **Retriever** – fetches relevant documents  
- **Parser** – processes outputs into usable data  

#### 2. Runnable Primitives (the controllers)
- **RunnableSequence** – chains runnables via `|`  
- **RunnableParallel** – runs multiple runnables in parallel  
- **RunnablePassthrough** – passes input unchanged  
- **RunnableLambda** – wraps a custom function  
- **RunnableBranch** – conditional routing (if‑else logic)  

---

## Key Takeaway
- **Task-Specific Runnables** do the actual AI work.  
- **Runnable Primitives** control how that work is orchestrated.  
- Together, they make LangChain pipelines **modular, standardized, and production-ready** for GenAI applications.

# Task-Specific Runnable: PromptTemplate

## Concept
PromptTemplate is a **task-specific Runnable** in LangChain that formats user input into a structured prompt before sending it to an LLM.

### Real-World Analogy
Think of PromptTemplate as a **form letter** or **template email**:
- Instead of writing every message from scratch, you prepare a template with placeholders:  
  *"Dear {Name}, thank you for attending {Event}..."*  
- Each time, you just fill in the blanks with the right values.

Similarly, PromptTemplate ensures that prompts are **consistent, reusable, and parameterized**.  
It acts like a **blueprint** for prompts, guaranteeing structure and clarity.

---

## Why It Matters
- Without PromptTemplate → developers manually concatenate strings (error-prone, messy).  
- With PromptTemplate → prompts are structured, reusable, and easy to maintain.  
- In web apps → ensures every user query is transformed into a clean format before hitting the LLM.

---

Imagine a teacher giving assignments:
- "Write about anything" → vague, inconsistent.  
- "Write a 200-word essay on {Topic}, with intro, body, and conclusion" → clear, structured.  

PromptTemplate does the same for LLMs.

In [1]:
# Code Example: PromptTemplate in LangChain

from langchain_core.prompts import PromptTemplate

# Step 1: Define a template with placeholders
template = "Write a short joke about {topic}."

# Step 2: Create a PromptTemplate Runnable
prompt = PromptTemplate.from_template(template)

# Step 3: Format the prompt with actual input
formatted_prompt = prompt.invoke({"topic": "cricket"})

print(formatted_prompt)

text='Write a short joke about cricket.'


### Where Task‑Specific Runnables Use the Runnable Interface

Even though **PromptTemplate, LLM/ChatModel, Retriever, and Parser** are conceptually different modules, in LangChain they are all wrapped as **Runnables**. This means they share a common interface and can be chained together seamlessly.

---

#### PromptTemplate
- Internally implements the Runnable interface.  
- You can call `.invoke({"topic": "cricket"})` instead of `.format()`.  
- This makes it behave like any other Runnable in a pipeline.



#### LLM / ChatModel
- Exposed as a Runnable.  
- Instead of `.predict()`, you use `.invoke("Tell me a joke")`.  
- This standardizes interaction across providers (OpenAI, Anthropic, Gemini).



#### Retriever
- Normally had `.get_relevant_documents(query)`.  
- As a Runnable, you just call `.invoke("machine learning")`.  
- This lets you plug it directly into a pipeline without custom glue code.



#### Parser
- Instead of `.parse(text)`, you use `.invoke(text)`.  
- This makes it chainable after an LLM Runnable.



#### Why This Matters
- **Before Runnables** → each component had its own method names (`.format()`, `.predict()`, `.parse()`, `.get_relevant_documents()`), making pipelines messy and inconsistent.  
- **With Runnables** → everything exposes `.invoke()`, `.batch()`, `.stream()`.  
- This uniformity allows Task‑Specific Runnables to slot directly into **Runnable Primitives** like `RunnableSequence` or `RunnableParallel`, enabling modular and production‑ready AI workflows.

In [4]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

# Step 1: Load environment variables
load_dotenv()  # Ensure your .env file contains GROQ_API_KEY

# Step 2: Task-Specific Runnables
prompt = PromptTemplate.from_template("Write a short joke about {topic}.")

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=1.5
)

parser = StrOutputParser()

# Step 3: Runnable Primitive - Sequence (using pipe operator)
chain = prompt | llm | parser

# Step 4: Invoke the chain
result = chain.invoke({"topic": "cricket"})
print(result)

Why did the cricket go to the doctor?

Because it had a bug in its system!

Hope that was a good delivery!


# Runnable Primitives (The Controllers)


### RunnableSequence

#### Concept
RunnableSequence is a **Runnable Primitive** that chains multiple Runnables together in a linear order.  
The output of one Runnable automatically becomes the input of the next.  

This solves the problem of manually wiring components together. Instead of writing glue code to pass outputs step‑by‑step, RunnableSequence provides a clean, standardized way to build pipelines.

---

#### Where It Can Be Used
- **Web Applications:** Streaming responses step‑by‑step (like ChatGPT’s token streaming).  
- **Classroom Demos:** Showing students how prompts flow into LLMs and then into parsers.  
- **Production Pipelines:** Connecting preprocessing → LLM → postprocessing without custom integration.  
- **Multi‑step Workflows:** Summarization, translation, or Q&A pipelines where each stage depends on the previous one.

---

#### Example Flow
`PromptTemplate → ChatModel → OutputParser`

This is the most common sequence:
1. PromptTemplate formats the input.  
2. LLM generates text.  
3. Parser structures the output.  

All three are Task‑Specific Runnables, and RunnableSequence orchestrates them.


In [5]:
# RunnableSequence Example with ChatGroq

import os
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

# Step 1: Load environment variables
load_dotenv()  # Ensure your .env file contains GROQ_API_KEY

# Step 2: Define Task-Specific Runnables
prompt = PromptTemplate.from_template("Write a short motivational quote about {topic}.")

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.7
)

parser = StrOutputParser()

# Step 3: RunnableSequence using pipe operator
chain = prompt | llm | parser

# Step 4: Invoke the chain
result = chain.invoke({"topic": "perseverance"})
print(result)


"Every setback is a stepping stone, not a roadblock. Keep pushing forward, for it's in the darkness of yesterday that the light of tomorrow is forged."


## RunnableParallel

### Concept
RunnableParallel is a **Runnable Primitive** that allows you to run multiple Runnables at the same time.  
Instead of executing them one after another, it executes them **concurrently** and returns all results together in a dictionary.



#### Where It Can Be Used
- **Web Applications:** When you need to query multiple models or services simultaneously to reduce latency.  
- **Classroom Demos:** Comparing outputs from different prompts or models side‑by‑side.  
- **Production Pipelines:** Running independent tasks (translation, summarization, sentiment analysis) in parallel.  
- **Evaluation Systems:** Sending the same input to multiple LLMs and aggregating their responses for benchmarking.



#### Example Flow
Input → RunnableParallel → { "english": pipeline_en, "french": pipeline_fr }  
- The same input sentence is sent to two pipelines.  
- One pipeline translates to English, the other to French.  
- Both results are returned together in a dictionary.


In [7]:
# RunnableParallel Example with ChatGroq

import os
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

# Step 1: Load environment variables
load_dotenv()  # Ensure your .env file contains GROQ_API_KEY

# Step 2: Define Task-Specific Runnables
prompt_en = PromptTemplate.from_template("Translate this sentence into English: {sentence}")
prompt_fr = PromptTemplate.from_template("Translate this sentence into French: {sentence}")

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

parser = StrOutputParser()

# Step 3: Build two pipelines (RunnableSequence via pipe operator)
pipeline_en = prompt_en | llm | parser
pipeline_fr = prompt_fr | llm | parser

# Step 4: RunnableParallel executes both pipelines at once
parallel = RunnableParallel({
    "english": pipeline_en,
    "french": pipeline_fr
})

# Step 5: Invoke the parallel tasks
result = parallel.invoke({"sentence": "Bonjour, comment ça va?"})
print(result)


{'english': 'The translation of the sentence "Bonjour, comment ça va?" is:\n\n"Hello, how are you?"\n\nThis is a common French greeting, where "bonjour" means "hello" or "good day," and "comment ça va?" is a question asking about the person\'s well-being or how they are doing.', 'french': 'The translation of the sentence "Bonjour, comment ça va?" into French is:\n\nBonjour, comment ça va?\n\nHowever, the more common way to ask "how are you?" in French is:\n\n- "Comment allez-vous?" (formal)\n- "Comment vas-tu?" (informal)\n\nSo, a more idiomatic translation of the sentence would be:\n\n"Bonjour, comment vas-tu?" (informal) or "Bonjour, comment allez-vous?" (formal)'}


In [9]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel
from pprint import pprint  # Pretty print for readable output

# Step 1: Load environment variables
load_dotenv()

# Step 2: Define Task-Specific Runnables
prompt_en = PromptTemplate.from_template("Translate this sentence into English: {sentence}")
prompt_fr = PromptTemplate.from_template("Translate this sentence into French: {sentence}")

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

parser = StrOutputParser()

# Step 3: Build two pipelines
pipeline_en = prompt_en | llm | parser
pipeline_fr = prompt_fr | llm | parser

# Step 4: RunnableParallel executes both pipelines at once
parallel = RunnableParallel({
    "english": pipeline_en,
    "french": pipeline_fr
})

# Step 5: Invoke the parallel tasks
result = parallel.invoke({"sentence": "Bonjour, comment ça va?"})

# Step 6: Pretty print the dictionary output
pprint(result, width=80)


{'english': 'The translation of the sentence "Bonjour, comment ça va?" is:\n'
            '\n'
            '"Hello, how are you?"\n'
            '\n'
            'This is a common French greeting, where "bonjour" means "hello" '
            'or "good day," and "comment ça va?" is a question asking about '
            "the person's well-being or how they are doing.",
 'french': 'The translation of the sentence "Bonjour, comment ça va?" into '
           'French is:\n'
           '\n'
           'Bonjour, comment ça va?\n'
           '\n'
           'However, the more common way to ask "how are you?" in French is:\n'
           '\n'
           '- "Comment allez-vous?" (formal)\n'
           '- "Comment vas-tu?" (informal)\n'
           '\n'
           'So, a more idiomatic translation of the sentence would be:\n'
           '\n'
           '"Bonjour, comment vas-tu?" (informal) or "Bonjour, comment '
           'allez-vous?" (formal)'}


In [11]:
# Loop through dictionary results and print cleanly
for key, value in result.items():
    print(f"\n--- {key.upper()} OUTPUT ---")
    print(value)


--- ENGLISH OUTPUT ---
The translation of the sentence "Bonjour, comment ça va?" is:

"Hello, how are you?"

This is a common French greeting, where "bonjour" means "hello" or "good day," and "comment ça va?" is a question asking about the person's well-being or how they are doing.

--- FRENCH OUTPUT ---
The translation of the sentence "Bonjour, comment ça va?" into French is:

Bonjour, comment ça va?

However, the more common way to ask "how are you?" in French is:

- "Comment allez-vous?" (formal)
- "Comment vas-tu?" (informal)

So, a more idiomatic translation of the sentence would be:

"Bonjour, comment vas-tu?" (informal) or "Bonjour, comment allez-vous?" (formal)


## RunnablePassthrough

#### Concept
RunnablePassthrough is a **Runnable Primitive** that simply forwards the input unchanged to the next step in the pipeline.  
It can also add extra keys to the input dictionary while keeping the original data intact.

---

#### Where It Can Be Used
- **Retrieval‑Augmented Generation (RAG):** Pass both the user’s original query and retrieved documents into the prompt.  
- **Context Preservation:** Ensure metadata (like user ID, session info, or timestamps) is carried forward without being lost.  
- **Debugging / Logging:** Keep the raw input available for inspection while still processing it downstream.  
- **Multi‑input Pipelines:** When you need to merge original input with additional computed values before sending to the LLM.

---

#### Example Flow
User Query → RunnablePassthrough → { "question": query, "docs": retrieved_docs } → PromptTemplate → LLM → Parser


In [16]:
# RunnablePassthrough Example

import os
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Step 1: Load environment variables
load_dotenv()

# Step 2: Define Task-Specific Runnables
prompt = PromptTemplate.from_template(
    "Answer the question using the context:\n\nQuestion: {question}\nContext: {context}"
)

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

parser = StrOutputParser()

# Step 3: RunnablePassthrough to carry both question and context
passthrough = RunnablePassthrough.assign(
    context=lambda x: "Cricket is a popular sport in India."
)

# Step 4: Build the pipeline
chain = passthrough | prompt | llm | parser

# Step 5: Invoke the chain
result = chain.invoke({"question": "Where is cricket popular?"})
print(result)

Cricket is popular in countries with a strong British colonial history, particularly in the Indian subcontinent, the Caribbean, and the United Kingdom. 

In the Indian subcontinent, cricket is extremely popular in countries such as:

1. India: As mentioned, cricket is a national obsession in India, with a massive following and a strong domestic league.
2. Pakistan: Cricket is the most popular sport in Pakistan, with a rich history of producing talented players.
3. Sri Lanka: Cricket is a beloved sport in Sri Lanka, with a strong domestic league and a passionate fan base.
4. Bangladesh: Cricket is gaining popularity in Bangladesh, with the national team competing in international tournaments.

In the Caribbean, cricket is popular in countries such as:

1. West Indies: The West Indies cricket team represents a group of Caribbean countries, including Barbados, Jamaica, and Trinidad and Tobago.
2. Jamaica: Cricket is a popular sport in Jamaica, with a strong domestic league and a rich hist

### RunnableLambda

#### Concept
RunnableLambda is a **Runnable Primitive** that lets you wrap a normal Python function (callable) into a Runnable.  
This makes custom logic (preprocessing, post‑processing, utility functions) compatible with the Runnable interface, so it can be chained with other Runnables using `|`.

---

#### Where It Can Be Used
- **Preprocessing:** Clean or normalize user input before sending it to an LLM.  
- **Post‑processing:** Format or filter LLM output before returning it to the user.  
- **Business Logic:** Insert custom rules (e.g., scoring, validation, logging) inside a pipeline.  
- **Classroom Demos:** Show how Python functions can be seamlessly integrated into LangChain pipelines.

---

#### Example Flow
User Input → RunnableLambda (custom function) → PromptTemplate → LLM → Parser

In [17]:
# RunnableLambda Example

import os
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

# Step 1: Load environment variables
load_dotenv()

# Step 2: Define a custom Python function
def clean_input(x: dict) -> dict:
    """Custom preprocessing: strip whitespace and capitalize."""
    return {"topic": x["topic"].strip().capitalize()}

# Step 3: Wrap the function as a RunnableLambda
clean_runnable = RunnableLambda(clean_input)

# Step 4: Define Task-Specific Runnables
prompt = PromptTemplate.from_template("Write a short motivational quote about {topic}.")
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.7
)
parser = StrOutputParser()

# Step 5: Build the pipeline with RunnableLambda
chain = clean_runnable | prompt | llm | parser

# Step 6: Invoke the chain
result = chain.invoke({"topic": " perseverance   "})
print(result)

"Every obstacle is a stepping stone, and every fall is an opportunity to rise stronger. Persevere, and you'll find your greatest strength lies not in your successes, but in your unwavering resolve."


**Backend Flow**

In [20]:
output = clean_input({"topic": " perseverance   "})
output

{'topic': 'Perseverance'}

{"topic": "Perseverance"}


-- PromptTemplate

"Write a short motivational quote about Perseverance."


**LLM**

"Perseverance is the key to success."

**Parser**

"Perseverance is the key to success."


**RunnableLambda acts like a wrapper: it doesn’t change your function, it just makes it “pipeline‑ready.”**

### RunnableBranch

#### Concept
RunnableBranch is a **Runnable Primitive** that enables conditional routing inside a pipeline.  
It works like an `if‑elif‑else` statement: based on the input, it decides which Runnable to execute.  
This allows you to build **adaptive workflows** where different logic is applied depending on the input type or content.

---

#### Where It Can Be Used
- **Domain Routing:** Send math questions to a calculator pipeline, and general questions to an LLM.  
- **Multi‑Model Systems:** Route queries about coding to one model, and creative writing to another.  
- **Business Logic:** Apply different workflows depending on user role (e.g., admin vs. customer).  
- **Classroom Demos:** Show students how conditional branching works in LangChain pipelines.

---

#### Example Flow
Input → RunnableBranch →  
- If input contains `"math"` → Math pipeline  
- Else → General LLM pipeline

In [23]:
# RunnableBranch Example (Corrected with proper default)

import os
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch

# Step 1: Load environment variables
load_dotenv()

# Step 2: Define Task-Specific Runnables
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)
parser = StrOutputParser()

# Pipeline for math queries
math_prompt = PromptTemplate.from_template("Solve this math problem: {question}")
math_chain = math_prompt | llm | parser

# Pipeline for general queries
general_prompt = PromptTemplate.from_template("Answer this question: {question}")
general_chain = general_prompt | llm | parser

# Step 3: Define RunnableBranch
branch = RunnableBranch(
    (lambda x: "math" in x["question"].lower(), math_chain),
    general_chain   # <-- default fallback (no condition)
)

# Step 4: Invoke the branch
result_math = branch.invoke({"question": "math: 12 + 8"})
print("--- MATH OUTPUT ---")
print(result_math)

result_general = branch.invoke({"question": "Who is the president of India?"})
print("\n--- GENERAL OUTPUT ---")
print(result_general)

--- MATH OUTPUT ---
To solve the math problem 12 + 8, I'll add the two numbers together.

12 + 8 = 20

--- GENERAL OUTPUT ---
As of my cut-off knowledge in 2023, the President of India is Droupadi Murmu. She took office on July 25, 2022. However, please note that my information may not be up to date, and I recommend verifying the current information for the most accurate answer.
